# 05 — Investigate one pixel

Give an **AOI number** and a **pixel id** (`pid`, read from `<aoi>_pixel_index.tif` in QGIS with *Identify*).
This notebook shows, for that pixel:

- **VH, VV and VH − VV** from the AOI's primary Sentinel-1 track: the single pixel (thin line) and its
  5x5 average (thick line, what the classifier uses). Triangles mark dates with more than 5 mm of rain
  in the previous 24 hours, which raises backscatter without any crop change.
- **NDVI and NDWI** from the exported Sentinel-2 images. **Dates where this pixel is cloudy are dropped**
  (clear score below `CLEAR_MIN`). Circles mark open water at the pixel (NDWI > 0).
- **What the draft model said**: class, rice probability, monsoon-rice probability.
- Grey bands: Jun–Jul (monsoon flooding), Oct–Nov (monsoon canopy / harvest), Feb (dry-season crop).

Run all cells. The first run for an AOI downloads its Sentinel-2 files (cached in `data/s2_reference/`).

## 1. Choose the pixel

In [ ]:
AOI = 146          # AOI number, e.g. 146
PID = 6697         # pixel id from <aoi>_pixel_index.tif
BACKEND = "plotly"     # "plotly" (interactive, hover for values) or "matplotlib" (static)
CLEAR_MIN = 60     # drop Sentinel-2 dates where this pixel's clear score is below this (0-100)
RAIN_MM = 5        # mark SAR dates with more rain than this in the previous 24 h

## 2. Gather the data

In [ ]:
import os, pathlib, warnings
warnings.filterwarnings("ignore")
# run from the repository root so config/ and processed/ resolve
root = pathlib.Path.cwd()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
os.chdir(root)

# Make sure THIS repository's package is imported, not another project's `sar_pipeline`
# that may be installed in the same Jupyter kernel. Select the "sar-rice-mapper (.venv)" kernel.
import sys
sys.path.insert(0, str(root / "src"))
import sar_pipeline
print("sar_pipeline loaded from:", pathlib.Path(sar_pipeline.__file__).parent)

import pandas as pd
from sar_pipeline.analysis import pixel_report as pr

result = pr.investigate(AOI, PID, clear_min=CLEAR_MIN, rain_mm=RAIN_MM)
pd.DataFrame(result["summary"].items(), columns=["", "value"]).set_index("")

## 3. Plot

In [ ]:
fig = pr.plot(result, backend=BACKEND)
if BACKEND == "plotly":
    fig.show()
else:
    import matplotlib.pyplot as plt
    plt.show()

## 4. The numbers behind the plot

Sentinel-1 acquisitions (dB):

In [ ]:
result["sar"].round(2)

Sentinel-2 dates. `kept = False` rows (cloudy or no data at this pixel) are not plotted:

In [ ]:
result["s2"].round(3)

## 5. Save (optional)

Uncomment to save the figure next to the notebook.

In [ ]:
# pr.plot(result, "matplotlib", f"aoi{AOI}_pid{PID}.png")
# pr.plot(result, "plotly", f"aoi{AOI}_pid{PID}.html")